<a href="https://colab.research.google.com/github/SiyumiJayawardhane/freshsense-imagemodel/blob/sithmi/Food_Spoilage_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Core data manipulation and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn for model selection, preprocessing, and evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Machine learning models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# For handling imbalanced datasets (Synthetic Minority Over-sampling Technique)
from imblearn.over_sampling import SMOTE

# For saving and loading models
import joblib

## 1. Setup and Data Loading

In [2]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

# Define the file path to the dataset
file_path = '/content/drive/MyDrive/TemperatureDataset.csv'
# Load the dataset into a pandas DataFrame
df = pd.read_csv(file_path)

# Print basic information about the loaded dataset
print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
print("First 5 rows of the dataset:")
# Rename columns for better readability and consistency
df.columns = ['Fruit', 'Temp', 'Humidity', 'MQ135', 'MQ3', 'Time']
# Display the first few rows of the DataFrame with new column names
print(df.head())

Mounted at /content/drive
Dataset loaded with 4558 rows and 6 columns.
First 5 rows of the dataset:
                                              Fruit  Temp  Humidity  MQ135  \
0  spoiled banana, spoiled cucumber, spoiled tomato  16.2      66.6      0   
1  spoiled banana, spoiled cucumber, spoiled tomato  16.2      67.8      0   
2  spoiled banana, spoiled cucumber, spoiled tomato  16.2      68.2      0   
3  spoiled banana, spoiled cucumber, spoiled tomato  16.3      67.3      0   
4  spoiled banana, spoiled cucumber, spoiled tomato  16.3      67.5      0   

    MQ3                 Time  
0  1188    4/6/2026 13:26:30  
1  1217  2026-04-06 13:28:44  
2  1157  2026-04-06 13:30:01  
3  1309  2026-04-06 13:45:01  
4  1337  2026-04-06 13:51:58  


In [3]:
# Display the initial dimensions (rows, columns) of the DataFrame
print("Initial shape:", df.shape)

Initial shape: (4558, 6)


## 2. Data Preprocessing and Feature Engineering

In [4]:
# Define a function to extract the main fruit type from the 'Fruit' column
def extract_fruit(text):
    # Convert text to lowercase for case-insensitive matching
    text = str(text).lower()
    # Check for specific fruit keywords
    if "banana" in text:
        return "banana"
    elif "cucumber" in text:
        return "cucumber"
    elif "tomato" in text:
        return "tomato"
    else:
        return "other" # Categorize as 'other' if no specific fruit is found

# Apply the function to the 'Fruit' column to standardize fruit names
df['Fruit'] = df['Fruit'].apply(extract_fruit)

In [5]:
# Define thresholds for MQ3 sensor readings to determine spoilage status
mq3_low = 800  # Threshold: below 800 indicates 'fresh'
mq3_high = 1200 # Threshold: 800-1199 indicates 'at-risk', 1200+ indicates 'spoiled'

# Define a function to assign a spoilage status based on the 'MQ3' sensor reading
def create_status(row):
    if row['MQ3'] >= mq3_high:
        return "spoiled"
    elif row['MQ3'] >= mq3_low:
        return "at-risk"
    else:
        return "fresh"

# Apply the 'create_status' function row-wise to create a new 'Status' column
df['Status'] = df.apply(create_status, axis=1)

In [6]:
# Define a robust function to parse datetime strings that might be in multiple formats
def parse_multiple_formats(series, formats):
    # Initialize a new series with NaT (Not a Time) values
    parsed_series = pd.Series(pd.NaT, index=series.index)
    for fmt in formats:
        # Identify rows that are still unparsed (NaT)
        unparsed_mask = parsed_series.isna()
        if unparsed_mask.any():
            # Attempt to parse unparsed rows with the current format
            parsed_series[unparsed_mask] = pd.to_datetime(
                series[unparsed_mask], format=fmt, errors='coerce' # 'coerce' turns unparseable dates into NaT
            )
    return parsed_series

# Define a list of possible datetime formats present in the 'Time' column
datetime_formats = [
    '%m/%d/%Y %H:%M:%S', # Format: Month/Day/Year Hour:Minute:Second
    '%Y-%m-%d %H:%M:%S', # Format: Year-Month-Day Hour:Minute:Second
    '%Y-%m-%d %H:%M:%S.%f' # Format: Year-Month-Day Hour:Minute:Second.Microsecond
]

# Apply the parsing function to the 'Time' column to convert it to datetime objects
df['Time'] = parse_multiple_formats(df['Time'], datetime_formats)

In [7]:
# Display concise summary of the DataFrame, including data types and non-null values
print(df.info())
# Display descriptive statistics for numerical columns
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4558 entries, 0 to 4557
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Fruit     4558 non-null   object        
 1   Temp      4558 non-null   float64       
 2   Humidity  4558 non-null   float64       
 3   MQ135     4558 non-null   int64         
 4   MQ3       4558 non-null   int64         
 5   Time      4558 non-null   datetime64[ns]
 6   Status    4558 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(2)
memory usage: 249.4+ KB
None
              Temp     Humidity        MQ135          MQ3  \
count  4558.000000  4558.000000  4558.000000  4558.000000   
mean     22.772049    74.204015    19.322071  1099.799473   
min      12.000000    50.600000     0.000000   308.000000   
25%      16.000000    67.825000     0.000000   741.000000   
50%      18.000000    74.500000     0.000000   962.000000   
75%      31.600000    79.8